In [4]:
#PREPARE RAG FORM AZURE WELL ARCHITETED FRAMEWORK

import unicodedata
from pathlib import Path

from pypdf import PdfReader
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

def prepare_index_pdf():

    embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        cache_folder="./models",
        device="cpu",
    )

    Settings.embed_model = embed_model
    Settings.chunk_size = 900
    Settings.chunk_overlap = 150

    def clean_text(text: str) -> str:
        text = text.replace("\x00", " ")
        text = unicodedata.normalize("NFKD", text)
        text = text.encode("ascii", errors="ignore").decode("ascii")
        return " ".join(text.split())

    pdf_path = Path("data/import_pdf/azure-well-architected-ai.pdf")
    storage_path = Path("data/storage/azure-well-architected-ai-local")

    reader = PdfReader(str(pdf_path))

    documents = []

    for page_number, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        text = clean_text(raw_text)

        if text:
            documents.append(
                Document(
                    text=text,
                    metadata={
                        "file_name": pdf_path.name,
                        "page": page_number,
                    },
                )
            )

    print(f"Page count: {len(documents)}")

    index = VectorStoreIndex.from_documents(
        documents,
        show_progress=True,
    )

    index.storage_context.persist(
        persist_dir=str(storage_path)
    )

    print(f"Index saved: {storage_path}")

prepare_index_pdf()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22685.47it/s]


Page count: 138


Generating embeddings: 100%|██████████| 138/138 [00:03<00:00, 36.27it/s]


Index saved: data/storage/azure-well-architected-ai-local


In [ ]:
## RUN MCP SERVER: python server.py

In [11]:
from pathlib import Path

ROOT = Path.cwd()
print(ROOT)
print(list(ROOT.iterdir()))

/Users/michalmolka/Downloads/python_llm_code/mcp_server
[PosixPath('/Users/michalmolka/Downloads/python_llm_code/mcp_server/server.py'), PosixPath('/Users/michalmolka/Downloads/python_llm_code/mcp_server/.DS_Store'), PosixPath('/Users/michalmolka/Downloads/python_llm_code/mcp_server/models'), PosixPath('/Users/michalmolka/Downloads/python_llm_code/mcp_server/__pycache__'), PosixPath('/Users/michalmolka/Downloads/python_llm_code/mcp_server/README.md'), PosixPath('/Users/michalmolka/Downloads/python_llm_code/mcp_server/prepare_index_llama.ipynb'), PosixPath('/Users/michalmolka/Downloads/python_llm_code/mcp_server/data')]


In [ ]:
import asyncio
import json
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


ROOT = Path.cwd()
print(ROOT)

async def main():
    server = StdioServerParameters(
        command=str(ROOT.parent / ".venv" / "bin" / "python"),
        args=[str(ROOT / "server.py")],
    )

    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            tools_json = json.dumps(tools.model_dump(), indent=2, ensure_ascii=False)
            print("=== MCP SERVER TOOLS ===")
            print(tools_json)
    return tools_json


mcp_tools_list = await main()

/Users/michalmolka/Downloads/python_llm_code/mcp_server
=== MCP SERVER TOOLS ===
{
  "meta": null,
  "nextCursor": null,
  "tools": [
    {
      "name": "retrieve_iowa_data",
      "title": null,
      "description": "\nRetrieve rows from one of the allowed Iowa parquet datasets.\n\nArgs:\n    file_name: Dataset name without extension. Allowed values are:\n        iowa_item, iowa_store, iowa_date, iowa_category, iowa_vendor.\n    limit: Maximum number of rows to return.\n\nReturns:\n    Dataset rows as JSON-serializable records.\n",
      "inputSchema": {
        "properties": {
          "file_name": {
            "enum": [
              "iowa_item",
              "iowa_store",
              "iowa_date",
              "iowa_category",
              "iowa_vendor"
            ],
            "title": "File Name",
            "type": "string"
          },
          "limit": {
            "default": 100,
            "title": "Limit",
            "type": "integer"
          }
        },
  

In [17]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

question = "Show me Iowa sales for whiskey."

prompt = f"""
You are a tool selector.

Available tools:
{mcp_tools_list}

User request:
{question}

Return only tool name.
"""

response = client.responses.create(
    model="openai/gpt-oss-20b",
    input=prompt
)

selected_tool = response.output_text.strip()

print("Selected tool:", selected_tool)

Selected tool: retrieve_iowa_data
